# IrOx OER small-sample screening
Compare an additive linear baseline and a Gaussian process using independent synthesis batches. The notebook opens with a separately generated synthetic dataset. The measured case and its validation scores are documented in the [case overview](README.md).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

case_dir = Path.cwd()
if not (case_dir / "benchmark.py").is_file():
    case_dir = case_dir / "examples" / "irox_screening"
if not (case_dir / "benchmark.py").is_file():
    raise FileNotFoundError("Open the notebook from the repository or example directory.")

sys.path.insert(0, str(case_dir.resolve()))
from benchmark import benchmark, write_report
from measurements import demo_inputs, load_inputs

root = case_dir.resolve().parents[1]
use_annotated_inputs = False
kind = "user_supplied" if use_annotated_inputs else "synthetic_demo"
if use_annotated_inputs:
    folder = root / "irox_private"
    inputs = load_inputs(folder / "batches.csv", folder / "scans.csv")
else:
    inputs = demo_inputs()

report, tables = benchmark(*inputs, dataset_kind=kind)
print(report["dataset_kind"])
display(pd.Series(report["counts"], name="count").to_frame())

## Repeatability and scan order

The nested model estimates variation between batches, between sessions of one batch and between scans in one session. A session ID is a proxy when physical electrode identity is missing. The first scan is the primary response. The batch mean tests whether the conclusion changes with response definition.

In [ ]:
components = {
    group: pd.DataFrame(result["components"]).T
    for group, result in report["repeatability"].items()
    if result["status"] == "estimated"
}
if components:
    display(pd.concat(components, names=["scope", "component"]).round(2))
else:
    print("The supplied data do not identify all three variance components.")

display(tables["scan_trends"]["pattern"].value_counts().rename("batches").to_frame())

## Hold out one synthesis at a time

Compare model errors on the same held-out batches, including the temperature subset. The full fit diagnostics go in summary.json. The diagonal in the scatter plot represents an exact held-out prediction.

In [ ]:
metrics = tables["metrics"]
display(metrics.round(3))
labels = {"ridge": "Ridge", "matern_gp": "Matérn GP"}
scopes = ["historical", "temperature_window"]
scope_labels = ["All historical", "350–500 °C"]
first = metrics.loc[metrics["target"].eq("eta_first_mv")]
rmse = first.pivot(index="scope", columns="model", values="rmse_mv").reindex(scopes)
chosen = "temperature_window" if "temperature_window" in first["scope"].values else "historical"
predictions = tables["loo_predictions"]
within = predictions.loc[predictions["scope"].eq(chosen) & predictions["target"].eq("eta_first_mv")]

fig, (error_ax, scatter_ax) = plt.subplots(1, 2, figsize=(10.5, 4.4), layout="constrained")
colors = ["#176B71", "#AD512F"]
rmse = rmse.reindex(columns=list(labels)).rename(columns=labels)
rmse.index = scope_labels
rmse.dropna(how="all").plot.bar(ax=error_ax, color=colors, rot=0, width=0.65)
error_ax.set(xlabel="", ylabel="LOOCV RMSE (mV)", title="First-scan model error")
for container in error_ax.containers:
    error_ax.bar_label(container, fmt="%.1f", padding=3, fontsize=9)
error_ax.set_ylim(0, rmse.max().max() * 1.2 + 1)
error_ax.legend(frameon=False)

start = within[["observed_mv", "predicted_mv"]].min().min() - 5
end = within[["observed_mv", "predicted_mv"]].max().max() + 5
scatter_ax.plot([start, end], [start, end], "--", color="0.55", linewidth=1)
for (model, label), color in zip(labels.items(), colors):
    subset = within.loc[within["model"].eq(model)]
    scatter_ax.scatter(subset["observed_mv"], subset["predicted_mv"], color=color, label=label, alpha=0.8)
scatter_ax.set(
    xlim=(start, end), ylim=(start, end), xlabel="Measured first-scan η₁₀ (mV)",
    ylabel="Held-out prediction (mV)", title="350–500 °C subset" if chosen == "temperature_window" else "Historical batches",
)
scatter_ax.legend(frameon=False)
plt.show()

## Compare measured recipes

These tables shortlist observed recipes, and show temperature pairs that match on the other recorded design factors. Treat small differences alongside repeatability. The tables support experimental review and do not imply a causal temperature or phase effect.

In [ ]:
columns = ["batch_id", "temperature_c", "pore_family", "eta_first_mv", "eta_mean_mv", "eta_sd_mv", "n_scans"]
display(tables["screened_observations"][columns].head(8).round(2))
display(tables["matched_temperature_pairs"].round(2))

write_report(report, tables, root / "irox_output" / kind)